In [0]:
#Função Info()

# Info dos Dados
# Tipo da Coluna
# Quantidade de linhas
# Quantidade de nulos
# Quantidade de Valores únicos
import pyspark.sql.functions as F
def info(x):

    # Número total de linhas
    n_rows = x.count()

    summary = []
    for c in x.columns:
        dtype = dict(x.dtypes)[c]
        n_nulls = x.filter(F.col(c).isNull()).count()
        n_uniques = x.select(c).distinct().count()
        summary.append((c, dtype, n_rows, n_nulls, n_uniques))

    # Crie o DataFrame de resumo
    summary_df = spark.createDataFrame(
        summary,
        ["coluna", "tipo", "qtd_linhas", "qtd_nulos", "qtd_valores_unicos"]
    )

    summary_df.show()

# The Transforming Logic

In [0]:
query = """

SELECT dim_p.product_key,
    dim_p.product_name,
    dim_c.customer_key,
    dim_c.first_name,
    crm_s.*
FROM workspace.silver.crm_sales_details AS crm_s

LEFT JOIN workspace.gold.dim_products AS dim_p
ON crm_s.product_number = dim_p.product_number
LEFT JOIN workspace.gold.dim_customers AS dim_c
ON crm_s.customer_id = dim_c.customer_id



"""

df = spark.sql(query)
display(df.limit(5))
info(df)

#df.groupBy(F.col("gender_final")).count().show()
#df.groupBy(F.col("gender")).count().show()
#df.groupBy(F.col("gender_erp")).count().show()


# Write into Golde Layer

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.gold.fact_sales")

In [0]:
%sql
SELECT *
FROM workspace.bronze.sales_details
LIMIT 10